In [408]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares

from uvx import UVXReader

In [409]:
POL_MAP = {
    "RR": -1,
    "LL": -2,
    "RL": -3,
    "LR": -4,
}

In [410]:
def brightness_temperature(S0_jy, theta_mas, freq_hz):
    """
    Observed-frame brightness temperature.
    """

    freq_ghz = freq_hz / 1e9
    Tb = (1.22e12 * S0_jy / (freq_ghz**2 * theta_mas**2))
    return Tb

In [411]:
def read_uvx(uvx_file, pol="RR", if_index=0, channel_index=0):

    with UVXReader(uvx_file) as uvx:

        data = uvx[0:len(uvx)]

        target_pol = POL_MAP[pol]

        stokes_index = None

        for i, s in enumerate(uvx.stokes):

            sval = s.value if hasattr(s, "value") else int(s)

            if sval == target_pol:
                stokes_index = i
                break

        if stokes_index is None:
            raise RuntimeError(
                f"{pol} not found"
            )

        u = data["header"]["u_wave"]
        v = data["header"]["v_wave"]

        baseline = (
            np.sqrt(u*u + v*v)
            * uvx.hdr.freq0
        )

        re = data["complex"]["re"][
            :, if_index, channel_index, stokes_index
        ]

        im = data["complex"]["im"][
            :, if_index, channel_index, stokes_index
        ]

        wt = data["complex"]["wt"][
            :, if_index, channel_index, stokes_index
        ]

        amp = np.sqrt(re**2 + im**2)

        good = (
            np.isfinite(baseline)
            & np.isfinite(amp)
            & np.isfinite(wt)
            & (wt > 0)
        )

        baseline = baseline[good]
        amp = amp[good]

        tl1 = data["header"]["tlsc1"][good]
        tl2 = data["header"]["tlsc2"][good]

        freq = uvx.hdr.freq0

    return (baseline, amp, tl1, tl2, freq)

In [412]:
def count_unique_baselines(tl1, tl2):
    pairs = set()

    for a, b in zip(tl1, tl2):
        pair = tuple(sorted((int(a), int(b))))
        pairs.add(pair)

    return len(pairs)

In [413]:
def gaussian_visibility(B, y0, S0, theta_rad):
    """
    Circular Gaussian visibility with constant offset.

    B         baseline in wavelengths
    y0        constant offset (Jy)
    S0        zero-spacing flux density (Jy)
    theta_rad FWHM (rad)
    """
    return y0 + S0 * np.exp(-((np.pi * theta_rad * B) ** 2) / (4.0 * np.log(2.0)))

In [414]:
def fit_gaussian(baseline, amp, sigma=None):
    # Сортируем
    order = np.argsort(baseline)
    baseline = baseline[order]
    amp = amp[order]
    if sigma is not None:
        sigma = sigma[order]

    # Определяем функцию невязок (взвешенных, если есть sigma)
    def residuals(params, B, y, sigma=None):
        y0, S0, theta = params
        model = gaussian_visibility(B, y0, S0, theta)
        if sigma is not None:
            return (y - model) / sigma   # взвешенные остатки
        else:
            return y - model

    # Начальные оценки
    y0_guess = 0.0
    S0_guess = np.max(amp)
    theta_guess = 1e-8

    # Границы
    bounds = ([0.0, 0.0, 0.0], [np.inf, np.inf, 1e-6])  # можно убрать верхнюю

    # Запускаем least_squares
    result = least_squares(
        residuals,
        x0=[y0_guess, S0_guess, theta_guess],
        args=(baseline, amp, sigma),
        bounds=bounds,
        method='trf',          # или 'lm' если без границ, но 'trf' с границами
        ftol=1e-12,
        xtol=1e-12,
        gtol=1e-12,
        max_nfev=100000,
        verbose=0,
    )

    y0_fit, S0_fit, theta_rad = result.x

    # Оценка ковариационной матрицы (приближённая)
    # Используем аппроксимацию Гессе из результата
    J = result.jac
    if J is not None:
        # Если есть якобиан, приблизим ковариацию
        res = result.fun
        # Для взвешенных остатков
        if sigma is not None:
            # уже взвешенные остатки, их дисперсия ~1
            cov = np.linalg.inv(J.T @ J) * (res @ res) / (len(res) - len(result.x))
        else:
            cov = np.linalg.inv(J.T @ J) * (res @ res) / (len(res) - len(result.x))
    else:
        cov = np.eye(3) * 1e-6

    # Извлекаем ошибки
    sigma_y0 = np.sqrt(np.abs(cov[0,0]))
    sigma_S0 = np.sqrt(np.abs(cov[1,1]))
    sigma_theta_rad = np.sqrt(np.abs(cov[2,2]))

    theta_mas = theta_rad * 206265000.0
    sigma_theta_mas = sigma_theta_rad * 206265000.0

    return (
        y0_fit, sigma_y0,
        S0_fit, sigma_S0,
        theta_rad, sigma_theta_rad,
        theta_mas, sigma_theta_mas,
        cov,
    )

In [415]:
def save_fit_plot(
    outfile,
    baseline,
    amp,
    y0,
    S0,
    theta_rad,
):
    xfit = np.linspace(baseline.min(), baseline.max(), 500)
    yfit = gaussian_visibility(xfit, y0, S0, theta_rad)   # теперь передаём y0, S0, theta_rad
    plt.figure(figsize=(8,6))
    plt.scatter(baseline, amp, s=10, label="data")
    plt.plot(xfit, yfit, lw=2, label="Gaussian fit")
    plt.xlabel("Baseline (lambda)")
    plt.ylabel("Correlated flux density (Jy)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

In [416]:
def brightness_temperature_error(Tb, S0, S0_err, theta, theta_err):

    frac = np.sqrt((S0_err / S0)**2 + (2.0 * theta_err / theta)**2)

    return Tb * frac

In [417]:
def parse_uvx_filename(path):

    name = os.path.basename(path)

    stem = name.replace(".uvx", "")

    parts = stem.split("_")

    mode = parts[0]
    session = parts[1]
    band = parts[2]
    date = parts[3]

    return {
        "mode": mode,
        "session": session,
        "band": band,
        "date": date,
    }

In [418]:
def choose_pol(band):

    if band in ["C", "K"]:
        return "LL"

    if band == "L":
        return "RR"
    
    if band == "P":
        return "RR"

    raise ValueError(
        f"Unknown band {band}"
    )

In [419]:
ROOT_DIR = r"i:\AGN\OJ287\CALIB_CLEAN"
OUT_DIR = "gaussian_fits"

os.makedirs(OUT_DIR, exist_ok=True)

rows = []

uvx_files = glob.glob(
    os.path.join(ROOT_DIR, "**", "*.uvx"),
    recursive=True,
)

print(f"Found {len(uvx_files)} UVX files")

for i, uvx_file in enumerate(uvx_files, start=1):
    name = os.path.basename(uvx_file)
    print(f"[{i}/{len(uvx_files)}] {name}")

    try:
        # ----------------------------------
        # parse filename
        # ----------------------------------
        meta = parse_uvx_filename(uvx_file)
        pol = choose_pol(meta["band"])

        if meta["mode"] == "GVLBI":
            mode_comment = "ground_only"
        else:
            mode_comment = "space_vlbi"

        # ----------------------------------
        # read uvx
        # ----------------------------------
        (
            baseline,
            amp,
            tl1,
            tl2,
            freq,
        ) = read_uvx(uvx_file, pol=pol)

        # ----------------------------------
        # count baselines
        # ----------------------------------
        nbase = count_unique_baselines(tl1, tl2)

        if nbase <= 1:
            rows.append([
                name,
                meta["mode"],
                meta["session"],
                meta["band"],
                meta["date"],
                nbase,
                "SKIPPED",
                np.nan,
                np.nan,
                np.nan,
                np.nan,
                np.nan,
                np.nan,
                "single_baseline",
            ])
            print(f"   SKIPPED ({nbase} baseline)")
            continue

        # ----------------------------------
        # gaussian fit with offset (y0)
        # ----------------------------------
        (
            y0_fit, sigma_y0,
            S0_fit, sigma_S0,
            theta_rad, sigma_theta_rad,
            theta_mas, sigma_theta_mas,
            pcov
        ) = fit_gaussian(baseline, amp)

        # Brightness temperature
        Tb = brightness_temperature(S0_fit, theta_mas, freq)
        Tb_err = brightness_temperature_error(
            Tb,
            S0_fit,
            sigma_S0,
            theta_mas,
            sigma_theta_mas,
        )

        # ----------------------------------
        # save figure
        # ----------------------------------
        png_name = os.path.splitext(name)[0] + ".png"
        png_file = os.path.join(OUT_DIR, png_name)

        save_fit_plot(png_file, baseline, amp, y0_fit, S0_fit, theta_rad)

        # Сохраняем в лог
        rows.append([
            name,
            meta["mode"],
            meta["session"],
            meta["band"],
            meta["date"],
            nbase,
            "OK",
            theta_mas,
            sigma_theta_mas,
            Tb,
            Tb_err,
            S0_fit,
            sigma_S0,
            mode_comment,
        ])

        print(
            f"   OK   "
            f"Nb={nbase}   "
            f"theta={theta_mas:.4f} mas   "
            f"Tb={Tb:.2e} K"
        )

    except Exception as e:
        rows.append([
            name,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            "FAILED",
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            str(e),
        ])
        print(f"   FAILED: {e}")

# --------------------------------------
# save log (эта часть у вас уже есть, я не меняю)
# --------------------------------------
log = pd.DataFrame(
    rows,
    columns=[
        "file",
        "mode",
        "session",
        "band",
        "date",
        "n_baselines",
        "status",
        "theta_mas",
        "theta_mas_err",
        "Tb_K",
        "Tb_K_err",
        "S0_Jy",
        "S0_Jy_err",
        "comment",
    ]
)

csv_file = os.path.join(OUT_DIR, "gaussian_fit_log.csv")
log.to_csv(csv_file, index=False)

print()
print("=" * 60)
print("Finished")
print(f"Results: {csv_file}")
print("=" * 60)

Found 78 UVX files
[1/78] RADIOASTRON_RAES03AA_C_20120427T213000_ASC_V4_cl_cl_ff_prd_tav_fav.uvx
   OK   Nb=5   theta=0.7143 mas   Tb=2.43e+11 K
[2/78] RADIOASTRON_RAES03AB_C_20120428T213000_ASC_V4_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=0.8469 mas   Tb=1.84e+11 K
[3/78] RADIOASTRON_RAES03FW_C_20121119T081000_ASC_V2_00_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=0.9622 mas   Tb=2.27e+11 K
[4/78] RADIOASTRON_RAES03FW_L_20121119T081000_ASC_V2_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=4.0546 mas   Tb=1.31e+11 K
[5/78] RADIOASTRON_RAES03P_C_20120404T183000_ASC_V2_00_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=0.2706 mas   Tb=2.43e+12 K
[6/78] RADIOASTRON_RAES03Q_C_20120405T170000_ASC_V2_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=0.2375 mas   Tb=3.15e+12 K
[7/78] GVLBI_RAES03RD_K_20130414T190000_ASC_V3_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=2.0510 mas   Tb=2.22e+09 K
[8/78] RADIOASTRON_RAES03RD_C_20130414T190000_ASC_V3_cl_ff_prd_tav_fav.uvx
   OK   Nb=6   theta=2.4064 mas   Tb=2.81e+1